# pandas II: agrupación, uniones y datos faltantes

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/iii-analisis-visualizacion-datos/13-pandas-agrupacion-uniones.ipynb)

## Trabajo previo

### Lecturas

Antes de la clase, revise las siguientes lecturas, en inglés. Los capítulos del libro de McKinney desarrollan los dos temas centrales de este cuaderno, la agrupación y las uniones; las dos lecciones del curso de Kaggle son breves, con ejercicios interactivos, y cubren la agrupación y los datos faltantes.

McKinney, W. (2022). Data aggregation and group operations. En *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media. https://wesmckinney.com/book/data-aggregation
\
\
McKinney, W. (2022). Data wrangling: Join, combine, and reshape. En *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media. https://wesmckinney.com/book/data-wrangling
\
\
Kaggle. (s. f.). Grouping and sorting. En *Pandas*. Kaggle Learn. Recuperado el 21 de setiembre de 2026, de https://www.kaggle.com/code/residentmario/grouping-and-sorting
\
\
Kaggle. (s. f.). Data types and missing values. En *Pandas*. Kaggle Learn. Recuperado el 21 de setiembre de 2026, de https://www.kaggle.com/code/residentmario/data-types-and-missing-values

### Otros recursos

En la [guía del usuario de pandas](https://pandas.pydata.org/docs/user_guide/index.html), los capítulos [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html), [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html) y [Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) son la referencia de los tres temas de este cuaderno.

## Introducción

Este cuaderno es la segunda parte de la serie *pandas*. La [parte I](https://gf0657-programacionsig.github.io/2026-ii/pandas-series-dataframes/) presentó las Series y los DataFrames, y las operaciones sobre **una tabla**: explorar, seleccionar, filtrar, ordenar y crear columnas. Esta parte responde tres preguntas que aparecen en cuanto se trabaja con datos reales:

- ¿Cómo se **resume** una tabla por categorías (ej. la población de cada provincia a partir de la de sus cantones)? Con la **agrupación**.
- ¿Cómo se **combinan** dos tablas que describen las mismas entidades (ej. los cantones y sus provincias)? Con las **uniones**.
- ¿Qué se hace cuando **faltan valores**? Detectarlos, entender por qué faltan y decidir, con criterio, si se descartan o se reemplazan.

En el camino se aprende a leer archivos de Excel —el formato en que muchas instituciones, como el INEC, publican sus datos— y, según el calendario de la lección de [asistentes de IA](https://gf0657-programacionsig.github.io/2026-ii/asistentes-ia/), esta semana se practica el uso de los asistentes para **generar y verificar** código de análisis de datos. La parte III de la serie se dedicará a los gráficos.

El hilo conductor siguen siendo los resultados de la [Estimación de Población y Vivienda 2022](https://inec.cr/) del Instituto Nacional de Estadística y Censos (INEC), ahora con sus tres niveles: provincias, cantones y distritos.

Como todos los cuadernos del curso, puede ejecutarse en la nube, con la insignia "Abrir en Google Colab", o localmente, en [Visual Studio Code](https://gf0657-programacionsig.github.io/2026-ii/vscode/) con el ambiente `geopython` como kernel. El archivo `.ipynb` puede descargarse con el botón de descarga (ícono de flecha hacia abajo) en la parte superior de la página del cuaderno, en el sitio web del curso.

In [1]:
import pandas as pd

# Dirección del directorio de datos del INEC en el repositorio del curso
DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/inec"

# Carga de los tres niveles de la división territorial
provincias = pd.read_csv(DATOS + "/provincias-2022.csv")
cantones = pd.read_csv(DATOS + "/cantones-2022.csv")
distritos = pd.read_csv(DATOS + "/distritos-2022.csv")

# Cantidad de filas de cada DataFrame
print(len(provincias), len(cantones), len(distritos))

7 82 487


## Agrupación

### Dividir, aplicar y combinar

Muchas preguntas sobre una tabla tienen la forma "¿cuánto vale *algo* **por cada** *categoría*?": la población por provincia, la cantidad de distritos por cantón, la densidad máxima por provincia. Todas se responden con la misma estrategia de tres pasos, conocida como **dividir, aplicar y combinar** (*split-apply-combine*):

1. **Dividir** las filas en grupos, según los valores de una columna.
2. **Aplicar** a cada grupo una función que lo resume en un valor (suma, promedio, máximo, conteo).
3. **Combinar** los resultados en una estructura nueva, con una fila por grupo.

Con las herramientas de la parte I, el primer paso se haría con un filtro por cada categoría. Por ejemplo, para una sola provincia:

In [2]:
# Población de la provincia de Guanacaste: filtro + suma
print(cantones[cantones["provincia"] == "Guanacaste"]["poblacion"].sum())

412808


Repetir esa línea para las siete provincias —o para los 82 cantones, si se agrupan distritos— sería el tipo de trabajo que la programación busca evitar. El método `groupby()` hace los tres pasos de una vez: recibe la columna que define los grupos, luego se selecciona la columna por resumir y, por último, se aplica la función:

In [3]:
# Población de cada provincia: dividir por provincia, aplicar sum() a poblacion
cantones.groupby("provincia")["poblacion"].sum()

provincia
Alajuela      1035464
Cartago        545092
Guanacaste     412808
Heredia        479117
Limón          470383
Puntarenas     500166
San José      1601167
Name: poblacion, dtype: int64

El resultado es una Series cuyo **índice son los grupos** (ordenados alfabéticamente). Como es una Series, admite todo lo visto en la parte I; por ejemplo, ordenarla:

In [4]:
# Las mismas sumas, de mayor a menor
cantones.groupby("provincia")["poblacion"].sum().sort_values(ascending=False)

provincia
San José      1601167
Alajuela      1035464
Cartago        545092
Puntarenas     500166
Heredia        479117
Limón          470383
Guanacaste     412808
Name: poblacion, dtype: int64

Las funciones de resumen (o de **agregación**) más usadas son `sum()`, `mean()`, `median()`, `min()`, `max()`, `count()` (cantidad de valores no nulos) y `size()` (cantidad de filas). Si se seleccionan varias columnas —con corchetes dobles, como siempre—, el resultado es un DataFrame:

In [5]:
# Totales de población y de viviendas por provincia
cantones.groupby("provincia")[["poblacion", "viviendas"]].sum()

,poblacion,viviendas
provincia,,
Alajuela,1035464,370120
Cartago,545092,183271
Guanacaste,412808,163747
Heredia,479117,168748
Limón,470383,171695
Puntarenas,500166,208991
San José,1601167,569463


In [6]:
# Cantidad de cantones por provincia: size() cuenta las filas de cada grupo
cantones.groupby("provincia").size()

provincia
Alajuela      16
Cartago        8
Guanacaste    11
Heredia       10
Limón          6
Puntarenas    11
San José      20
dtype: int64

(`size()` sobre los grupos equivale al `value_counts()` de la parte I, salvo por el orden del resultado.)

### Varias funciones a la vez: agg()

El método `agg()` aplica varias funciones de una vez. Su forma más legible es la **agregación con nombre**, en la que cada argumento define una columna del resultado con un par `(columna de origen, función)`:

In [7]:
# Resumen por provincia, con una columna por cada agregación
resumen = cantones.groupby("provincia").agg(
    cantones=("canton", "size"),
    poblacion=("poblacion", "sum"),
    area_km2=("area_km2", "sum"),
    densidad_maxima=("densidad", "max")
)

resumen

,cantones,poblacion,area_km2,densidad_maxima
provincia,,,,
Alajuela,16,1035464,9772.13,1005.3
Cartago,8,545092,3093.23,2253.0
Guanacaste,11,412808,10196.30,80.5
Heredia,10,479117,2663.31,3580.3
Limón,6,470383,9176.88,74.6
Puntarenas,11,500166,11275.00,160.2
San José,20,1601167,4969.77,9019.6


Note que `provincia` no es una columna del resultado, sino su **índice**. Para volver a tenerla como columna —lo que hará falta para las uniones de la sección siguiente— se usa `reset_index()`, que convierte el índice en columna y restablece el índice de posiciones:

In [8]:
resumen = resumen.reset_index()

resumen

,provincia,cantones,poblacion,area_km2,densidad_maxima
0,Alajuela,16,1035464,9772.13,1005.3
1,Cartago,8,545092,3093.23,2253.0
2,Guanacaste,11,412808,10196.30,80.5
3,Heredia,10,479117,2663.31,3580.3
4,Limón,6,470383,9176.88,74.6
5,Puntarenas,11,500166,11275.00,160.2
6,San José,20,1601167,4969.77,9019.6


### Una trampa: no todo se puede promediar

`groupby()` calcula lo que se le pida, tenga sentido o no. ¿Cuál es la densidad de población de cada provincia? Una respuesta tentadora es promediar la densidad de sus cantones:

In [9]:
# Promedio de las densidades de los cantones de cada provincia... ¿es la densidad de la provincia?
cantones.groupby("provincia")["densidad"].mean().round(1)

provincia
Alajuela       264.2
Cartago        480.5
Guanacaste      40.3
Heredia       1453.7
Limón           57.6
Puntarenas      62.2
San José      2046.6
Name: densidad, dtype: float64

Según ese cálculo, Heredia tendría más de 1400 habitantes por km². La densidad correcta es la población total entre el área total, y ambos totales ya están en `resumen`:

In [10]:
# Densidad de cada provincia: población total / área total
resumen["densidad"] = (resumen["poblacion"] / resumen["area_km2"]).round(1)

resumen[["provincia", "poblacion", "area_km2", "densidad"]]

,provincia,poblacion,area_km2,densidad
0,Alajuela,1035464,9772.13,106.0
1,Cartago,545092,3093.23,176.2
2,Guanacaste,412808,10196.30,40.5
3,Heredia,479117,2663.31,179.9
4,Limón,470383,9176.88,51.3
5,Puntarenas,500166,11275.00,44.4
6,San José,1601167,4969.77,322.2


La densidad real de Heredia es de unos 180 habitantes por km², ocho veces menor. El promedio simple trata igual a todos los cantones, pero Heredia tiene varios cantones pequeños y muy densos (como Flores o San Pablo) y uno enorme y casi despoblado (Sarapiquí), que pesa lo mismo que los demás en el promedio aunque ocupa la mayor parte del área. La regla general: las **razones** (densidades, porcentajes, tasas) no se promedian; se recalculan a partir de los totales. Este resultado puede verificarse contra una fuente independiente: el archivo de provincias del INEC.

In [11]:
# Verificación contra la densidad publicada por el INEC
provincias[["provincia", "densidad"]]

,provincia,densidad
0,San José,322.2
1,Alajuela,106.0
2,Cartago,176.2
3,Heredia,179.9
4,Guanacaste,40.5
5,Puntarenas,44.4
6,Limón,51.3


### Agrupar por más de una columna

Los grupos pueden definirse con varias columnas, en una lista. Es necesario, por ejemplo, para agrupar los distritos por cantón con seguridad. En Costa Rica ningún nombre de cantón se repite, pero los de distrito sí, y mucho:

In [12]:
# Nombres de distrito más repetidos del país
distritos["distrito"].value_counts().head()

distrito
San Rafael     12
San Isidro      9
San Antonio     8
San Juan        7
San Pedro       7
Name: count, dtype: int64

Por eso las entidades se agrupan (y se unen) por su **código**, que es único, y no por su nombre. Para conservar el nombre en el resultado, se agrupa por ambas columnas:

In [13]:
# Cantidad de distritos y distrito más poblado de cada cantón
distritos.groupby(["codigo_canton", "canton"]).agg(
    distritos=("distrito", "size"),
    poblacion_mayor_distrito=("poblacion", "max")
).reset_index().head()

,codigo_canton,canton,distritos,poblacion_mayor_distrito
0,101,San José,11,83573
1,102,Escazú,3,32361
2,103,Desamparados,13,38194
3,104,Puriscal,9,12189
4,105,Tarrazú,3,10674


*Ejercicios de esta sección: ejercicios sobre agrupación, en la sección de ejercicios al final del cuaderno.*

## Lectura de archivos de Excel

Los archivos CSV del curso se prepararon a partir de un [archivo de Excel del INEC](https://admin.inec.cr/sites/default/files/2023-11/reResultadosEstimacionPoblacionVivienda2022_3.xlsx) con 12 cuadros, uno por hoja. Excel es el formato en que muchas instituciones publican sus datos, y sus cuadros están pensados para ser **leídos por personas**, no por programas: tienen títulos, encabezados de varias filas, filas en blanco, totales y notas al pie. La tabla 1 muestra el contenido del cuadro 4, con la población de las provincias en 2011 y en 2022.

<figure style="text-align: center; margin: 20px 0;">
    <figcaption><strong>Tabla 1</strong>. Contenido de la hoja "4" del archivo de Excel del INEC, con el número de fila de la hoja de cálculo. Fuente: Instituto Nacional de Estadística y Censos (2023).</figcaption>
    <table class="table table-bordered table-striped" style="margin: 0 auto;">
    <thead>
        <tr><th>Fila</th><th>A</th><th>B</th><th>C</th><th>D</th></tr>
    </thead>
    <tbody>
        <tr><td>1</td><td>CUADRO 4</td><td></td><td></td><td></td></tr>
        <tr><td>2</td><td>Costa Rica: Población total y tasa de crecimiento, según provincia, 2011 - 2022</td><td></td><td></td><td></td></tr>
        <tr><td>3</td><td></td><td></td><td></td><td></td></tr>
        <tr><td>4</td><td>Provincia</td><td>Total</td><td></td><td>Tasa de crecimiento</td></tr>
        <tr><td>5</td><td></td><td>2011</td><td>2022</td><td></td></tr>
        <tr><td>6</td><td></td><td></td><td></td><td></td></tr>
        <tr><td>7</td><td>Costa Rica</td><td>4301712</td><td>5044197</td><td>1.45</td></tr>
        <tr><td>8</td><td></td><td></td><td></td><td></td></tr>
        <tr><td>9</td><td>San José</td><td>1404242</td><td>1601167</td><td>1.19</td></tr>
        <tr><td>...</td><td>...</td><td>...</td><td>...</td><td>...</td></tr>
        <tr><td>15</td><td>Limón</td><td>386862</td><td>470383</td><td>1.78</td></tr>
        <tr><td>16</td><td>1/ Tasa de crecimiento promedio anual por cien.</td><td></td><td></td><td></td></tr>
        <tr><td>17</td><td>Fuente: INEC-Costa Rica. Censo Nacional de Población 2011 y Estimación de Población y Vivienda 2022.</td><td></td><td></td><td></td></tr>
    </tbody>
    </table>
</figure>

La función `pd.read_excel()` lee una hoja de un archivo de Excel y, como `read_csv()`, acepta una URL. Sus argumentos permiten quedarse solo con el rectángulo de datos:

- `sheet_name`: nombre de la hoja (aquí, `"4"`).
- `skiprows`: cantidad de filas que se omiten al inicio (las 8 primeras: títulos, encabezados y el total del país).
- `nrows`: cantidad de filas de datos que se leen (las 7 provincias; así se excluyen las notas al pie).
- `header=None` y `names`: indican que no hay fila de encabezados aprovechable y asignan nombres de columna propios, con las convenciones del curso.

In [14]:
# Lectura del cuadro 4: población de las provincias en 2011 y 2022
censo_2011 = pd.read_excel(
    DATOS + "/reResultadosEstimacionPoblacionVivienda2022_3.xlsx",
    sheet_name="4",
    skiprows=8,
    nrows=7,
    header=None,
    names=["provincia", "poblacion_2011", "poblacion_2022", "tasa_crecimiento"]
)

censo_2011

,provincia,poblacion_2011,poblacion_2022,tasa_crecimiento
0,San José,1404242,1601167,1.19
1,Alajuela,848146,1035464,1.81
2,Cartago,490903,545092,0.95
3,Heredia,433677,479117,0.91
4,Guanacaste,326953,412808,2.12
5,Puntarenas,410929,500166,1.79
6,Limón,386862,470383,1.78


Para leer archivos `.xlsx`, pandas requiere el paquete [openpyxl](https://openpyxl.readthedocs.io/), ya instalado en Colab y en el ambiente `geopython`. Antes de escribir un llamado a `read_excel()` conviene abrir el archivo en una hoja de cálculo y anotar en cuál fila empiezan y terminan los datos; después, siempre hay que revisar el resultado (`shape`, `dtypes`, primeras y últimas filas) para confirmar que se leyó el rectángulo correcto.

*Ejercicios de esta sección: ejercicios sobre lectura de Excel, en la sección de ejercicios al final del cuaderno.*

## Uniones

Una **unión** (*merge* o *join*) combina dos DataFrames en uno, emparejando las filas que tienen el mismo valor en una columna común, llamada **llave** (*key*). Es la operación que permite relacionar tablas que describen las mismas entidades desde fuentes o niveles distintos, y es la misma idea de las uniones de tablas de los SIG y de las bases de datos: en la unidad de datos geoespaciales, esta misma operación unirá una capa de cantones con tablas de datos para elaborar mapas.

### Unir por una llave común

¿Qué porcentaje de la población de su provincia tiene cada cantón? La población del cantón está en `cantones`, y la de la provincia, en `provincias`. La llave común es `codigo_provincia`. El método `merge()` agrega a cada cantón las columnas de su provincia:

In [15]:
# A cada cantón se le agrega la población de su provincia
cantones_provincia = cantones.merge(
    provincias[["codigo_provincia", "poblacion"]],
    on="codigo_provincia",
    suffixes=("", "_provincia")
)

cantones_provincia[["provincia", "canton", "poblacion", "poblacion_provincia"]].head()

,provincia,canton,poblacion,poblacion_provincia
0,San José,San José,352381,1601167
1,San José,Escazú,71500,1601167
2,San José,Desamparados,223226,1601167
3,San José,Puriscal,38525,1601167
4,San José,Tarrazú,17810,1601167


Tres detalles del llamado:

- Del DataFrame de la derecha se seleccionaron solo las columnas necesarias: la llave y la que se quiere agregar. Es una buena práctica, porque evita duplicar las demás.
- `on` indica la llave. Esta es una unión de **muchos a uno**: muchos cantones se emparejan con la misma provincia, cuya población se repite en cada uno.
- Como ambos DataFrames tienen una columna `poblacion`, pandas debe distinguirlas en el resultado: `suffixes` indica qué agregar al nombre de la columna de la izquierda (nada) y al de la derecha (`_provincia`).

Con las dos columnas en la misma tabla, el cálculo es una operación vectorizada:

In [16]:
# Porcentaje de la población provincial que reside en cada cantón
cantones_provincia["porcentaje_provincia"] = (
    cantones_provincia["poblacion"] / cantones_provincia["poblacion_provincia"] * 100
).round(1)

# Los cinco cantones con más peso en su provincia
cantones_provincia.sort_values("porcentaje_provincia", ascending=False)[
    ["provincia", "canton", "porcentaje_provincia"]
].head()

,provincia,canton,porcentaje_provincia
77,Limón,Pococí,31.1
20,Alajuela,Alajuela,31.1
36,Cartago,Cartago,30.3
65,Puntarenas,Puntarenas,28.3
44,Heredia,Heredia,27.5


### Agrupar y unir: verificación entre niveles

La agrupación y la unión se combinan con frecuencia. Por ejemplo, para verificar la consistencia de los datos: ¿la suma de las poblaciones de los distritos de cada cantón coincide con la población del cantón? Primero se agrupa y luego se une el resultado, por el código del cantón:

In [17]:
# 1. Agrupación: población de cada cantón según la suma de sus distritos
suma_distritos = distritos.groupby("codigo_canton")["poblacion"].sum().reset_index()

# 2. Unión con los cantones, por el código
verificacion = cantones[["codigo_canton", "canton", "poblacion"]].merge(
    suma_distritos,
    on="codigo_canton",
    suffixes=("", "_distritos")
)

# 3. ¿Coinciden las dos poblaciones en los 82 cantones?
print((verificacion["poblacion"] == verificacion["poblacion_distritos"]).all())

True


### Tipos de unión

En los dos ejemplos anteriores, cada llave de un DataFrame tenía pareja en el otro. En los datos reales eso no siempre ocurre: una tabla puede tener entidades que la otra no tiene. El argumento `how` de `merge()` decide qué pasa con las filas cuya llave está en un solo DataFrame, y sus cuatro valores se ilustran en la figura 1 como **diagramas de Venn**: cada círculo es un DataFrame, la intersección son las llaves presentes en ambos y la región sombreada son las filas que quedan en el resultado.

<figure style="text-align: center;">
  <img
    src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iii-analisis-visualizacion-datos/img/tipos-union.png"
    alt="Cuatro diagramas de Venn con dos círculos, izquierda y derecha, que sombrean las filas que conserva cada tipo de unión: inner solo la intersección, left todo el círculo izquierdo, right todo el círculo derecho y outer ambos círculos"
    style="max-width: 720px; width: 100%;"
  >
  <figcaption><strong>Figura 1</strong>. Tipos de unión de <code>merge()</code> según el argumento <code>how</code>. El área sombreada indica las filas que quedan en el resultado. Elaboración propia.</figcaption>
</figure>

Para ver los cuatro tipos con datos, se construyen dos DataFrames pequeños a partir de `provincias`: `izquierda`, con la población de las cinco primeras provincias (códigos 1 a 5), y `derecha`, con el área de las cuatro últimas (códigos 4 a 7). Solo Heredia y Guanacaste (códigos 4 y 5) están en ambos.

In [18]:
# Dos DataFrames que solo comparten dos llaves (códigos 4 y 5)
izquierda = provincias[["codigo_provincia", "provincia", "poblacion"]].head(5)
derecha = provincias[["codigo_provincia", "area_km2"]].tail(4)

print(izquierda)
print()
print(derecha)

   codigo_provincia   provincia  poblacion
0                 1    San José    1601167
1                 2    Alajuela    1035464
2                 3     Cartago     545092
3                 4     Heredia     479117
4                 5  Guanacaste     412808

   codigo_provincia  area_km2
3                 4   2663.30
4                 5  10196.29
5                 6  11275.00
6                 7   9176.88


`how="inner"` (el valor por defecto) conserva **solo las llaves presentes en ambos** DataFrames: la intersección de los círculos. Las demás filas desaparecen, sin ningún aviso:

In [19]:
# inner: solo Heredia y Guanacaste, que están en ambos
izquierda.merge(derecha, on="codigo_provincia", how="inner")

,codigo_provincia,provincia,poblacion,area_km2
0,4,Heredia,479117,2663.30
1,5,Guanacaste,412808,10196.29


`how="left"` conserva **todas las filas del DataFrame de la izquierda**, tengan pareja o no. Las que no la tienen reciben `NaN` en las columnas que venían de la derecha:

In [20]:
# left: las cinco provincias de la izquierda; tres quedan sin área
izquierda.merge(derecha, on="codigo_provincia", how="left")

,codigo_provincia,provincia,poblacion,area_km2
0,1,San José,1601167,NaN
1,2,Alajuela,1035464,NaN
2,3,Cartago,545092,NaN
3,4,Heredia,479117,2663.30
4,5,Guanacaste,412808,10196.29


`how="right"` es el caso simétrico (todas las filas de la derecha) y `how="outer"` conserva **todas las filas de ambos**: la unión de los dos círculos. El argumento `indicator=True` agrega una columna `_merge` que dice, para cada fila, en cuál región del diagrama está: `both` (en ambos), `left_only` (solo en la izquierda) o `right_only` (solo en la derecha).

In [21]:
# outer: las siete provincias; _merge indica de dónde viene cada fila
izquierda.merge(derecha, on="codigo_provincia", how="outer", indicator=True)

,codigo_provincia,provincia,poblacion,area_km2,_merge
0,1,San José,1601167.0,NaN,left_only
1,2,Alajuela,1035464.0,NaN,left_only
2,3,Cartago,545092.0,NaN,left_only
3,4,Heredia,479117.0,2663.30,both
4,5,Guanacaste,412808.0,10196.29,both
5,6,NaN,NaN,11275.00,right_only
6,7,NaN,NaN,9176.88,right_only


Las filas que solo estaban en `derecha` (Puntarenas y Limón) no tienen nombre ni población, porque esas columnas venían de `izquierda`. Una comprobación rápida de cuántas filas produce cada tipo de unión:

In [22]:
# Cantidad de filas del resultado según el tipo de unión
for tipo in ["inner", "left", "right", "outer"]:
    resultado = izquierda.merge(derecha, on="codigo_provincia", how=tipo)
    print(f'how="{tipo}": {len(resultado)} filas')

how="inner": 2 filas
how="left": 5 filas
how="right": 4 filas
how="outer": 7 filas


La tabla 2 resume los cuatro tipos y cuándo se usa cada uno. En la práctica, la elección más frecuente es `how="left"`, con la tabla que se quiere **enriquecer** a la izquierda: se le agregan columnas sin perder ninguna de sus filas, y las llaves sin pareja quedan a la vista como valores faltantes. `how="inner"` es adecuada cuando solo interesan las entidades con datos en ambas tablas, pero hay que recordar que descarta filas sin avisar. `how="outer"` con `indicator=True` es la herramienta para **diagnosticar** una unión: muestra qué llaves fallaron y de cuál lado.

<figure style="text-align: center; margin: 20px 0;">
    <figcaption><strong>Tabla 2</strong>. Tipos de unión de <code>merge()</code>. Elaboración propia.</figcaption>
    <table class="table table-bordered table-striped" style="margin: 0 auto;">
    <thead>
        <tr><th><code>how</code></th><th>Filas del resultado</th><th>Uso típico</th></tr>
    </thead>
    <tbody>
        <tr><td><code>"inner"</code> (por defecto)</td><td>Llaves presentes en ambos DataFrames</td><td>Quedarse solo con las entidades que tienen datos en las dos tablas</td></tr>
        <tr><td><code>"left"</code></td><td>Todas las de la izquierda</td><td>Agregar columnas a una tabla sin perder filas</td></tr>
        <tr><td><code>"right"</code></td><td>Todas las de la derecha</td><td>Lo mismo, con la tabla principal a la derecha (poco usado: se prefiere invertir el orden y usar <code>"left"</code>)</td></tr>
        <tr><td><code>"outer"</code></td><td>Todas las de ambos</td><td>Diagnosticar qué llaves no coinciden, con <code>indicator=True</code></td></tr>
    </tbody>
    </table>
</figure>

### Llaves que no coinciden

¿Cuánto creció la población de cada provincia entre 2011 y 2022? La población de 2011 está en `censo_2011`, el cuadro leído de Excel, que no tiene códigos: la única llave posible es el **nombre** de la provincia. Se une con `how="left"`, la opción segura para enriquecer `provincias` sin perder filas:

In [23]:
# Unión por nombre de provincia, conservando todas las provincias de la izquierda
crecimiento = provincias[["provincia", "poblacion"]].merge(
    censo_2011[["provincia", "poblacion_2011"]],
    on="provincia",
    how="left"
)

crecimiento

,provincia,poblacion,poblacion_2011
0,San José,1601167,1404242.0
1,Alajuela,1035464,848146.0
2,Cartago,545092,490903.0
3,Heredia,479117,433677.0
4,Guanacaste,412808,326953.0
5,Puntarenas,500166,NaN
6,Limón,470383,386862.0


La fila de Puntarenas quedó sin población de 2011: en su lugar aparece `NaN` (*not a number*), el marcador de pandas para un **valor faltante**. Note además que toda la columna pasó de enteros a números de punto flotante (`1404242.0`): `NaN` es un valor de punto flotante, y una columna tiene un solo tipo.

¿Por qué no coincidió Puntarenas, si está en ambas tablas? Al desplegar un DataFrame no se nota, pero la lista de valores revela el problema:

In [24]:
# Los nombres de provincia del cuadro de Excel, como lista
censo_2011["provincia"].tolist()

['San José',
 'Alajuela',
 'Cartago',
 'Heredia',
 'Guanacaste',
 'Puntarenas ',
 'Limón']

En el archivo del INEC, el nombre está escrito como `'Puntarenas '`, con un **espacio al final**. Para una persona es la misma provincia; para `merge()`, que compara las hileras carácter por carácter, son llaves distintas. Los espacios sobrantes, las diferencias de mayúsculas y las tildes ausentes son la causa más común de uniones incompletas, y una razón más para unir por código cuando lo hay. Con `how="inner"` habría sido peor: Puntarenas habría desaparecido del resultado sin ningún aviso.

La solución es limpiar la llave antes de unir. Los métodos de hileras de la serie *Fundamentos de Python* están disponibles para columnas completas mediante el accesor `.str`:

In [25]:
# strip() elimina los espacios al inicio y al final de cada valor de la columna
censo_2011["provincia"] = censo_2011["provincia"].str.strip()

# Se repite la unión, ahora con las llaves limpias
crecimiento = provincias[["provincia", "poblacion"]].merge(
    censo_2011[["provincia", "poblacion_2011"]],
    on="provincia",
    how="left"
)

# Aumento de población entre 2011 y 2022
crecimiento["aumento"] = crecimiento["poblacion"] - crecimiento["poblacion_2011"]

crecimiento.sort_values("aumento", ascending=False)

,provincia,poblacion,poblacion_2011,aumento
0,San José,1601167,1404242,196925
1,Alajuela,1035464,848146,187318
5,Puntarenas,500166,410929,89237
4,Guanacaste,412808,326953,85855
6,Limón,470383,386862,83521
2,Cartago,545092,490903,54189
3,Heredia,479117,433677,45440


### Ejemplo: casos de COVID-19 por cantón

Un caso típico de unión en el trabajo geográfico: los datos de un fenómeno vienen de una institución y la población de referencia, de otra. El [Ministerio de Salud](https://www.ministeriodesalud.go.cr/) publicó durante la pandemia el acumulado de casos positivos de COVID-19 por cantón; el archivo con los datos al 30 de mayo de 2022 está en el [directorio de datos](https://github.com/gf0657-programacionsig/2026-ii/tree/main/datos/ministerio-salud) del curso. Para comparar cantones de tamaños muy distintos, la pregunta no es cuántos casos hubo, sino cuántos por cada 100 000 habitantes: una **tasa**, y la población está en `cantones`, del INEC.

El archivo del Ministerio no está en el formato limpio de los CSV del curso: los campos se separan con punto y coma, la codificación de caracteres es la de Windows (`cp1252`, no UTF-8) y tiene una columna por cada fecha, más de 800. `read_csv()` tiene un argumento para cada particularidad: `sep`, `encoding` y `usecols`, que lee solo las columnas indicadas.

In [26]:
# Dirección del directorio de datos del Ministerio de Salud en el repositorio del curso
DATOS_SALUD = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/ministerio-salud"

# Casos positivos acumulados por cantón al 30 de mayo de 2022
covid = pd.read_csv(
    DATOS_SALUD + "/05_30_22_CSV_POSITIVOS.csv",
    sep=";",
    encoding="cp1252",
    usecols=["cod_canton", "canton", "30/05/2022"]
)

print(covid.shape)
covid.tail(3)

(84, 3)


,cod_canton,canton,30/05/2022
81,704.0,Talamanca,5468.0
82,999.0,Otros,352.0
83,NaN,NaN,NaN


La revisión de las últimas filas muestra dos cosas que no aparecen en los cantones del INEC: una fila `Otros` (código 999, casos sin cantón asignado) y una fila completamente vacía al final del archivo. Por esa fila vacía, pandas leyó los códigos como números de punto flotante (`704.0`). Antes de unir se descarta la fila vacía, se convierten los códigos a enteros y, con `rename()`, se ponen a las columnas los nombres que usa el resto del cuaderno, de modo que la llave se llame igual en ambos DataFrames:

In [27]:
# Limpieza: sin la fila vacía, códigos enteros y nombres de columna del curso
covid = covid.dropna(subset=["cod_canton"])
covid["cod_canton"] = covid["cod_canton"].astype(int)
covid["30/05/2022"] = covid["30/05/2022"].astype(int)
covid = covid.rename(columns={"cod_canton": "codigo_canton", "30/05/2022": "positivos"})

covid.tail(3)

,codigo_canton,canton,positivos
80,703,Siquirres,10349
81,704,Talamanca,5468
82,999,Otros,352


Antes de calcular nada, un diagnóstico de la unión con `how="outer"` e `indicator=True`: ¿qué llaves están en ambos DataFrames y cuáles solo en uno?

In [28]:
# Diagnóstico: ¿en cuál región del diagrama cae cada cantón?
diagnostico = cantones[["codigo_canton", "canton"]].merge(
    covid,
    on="codigo_canton",
    how="outer",
    indicator=True,
    suffixes=("", "_salud")
)

print(diagnostico["_merge"].value_counts())

# Las filas que no están en ambos
diagnostico[diagnostico["_merge"] != "both"]

_merge
both          82
right_only     1
left_only      0
Name: count, dtype: int64


,codigo_canton,canton,canton_salud,positivos,_merge
82,999,NaN,Otros,352,right_only


Los 82 cantones del INEC tienen pareja en los datos del Ministerio (`both`), y la única fila sin pareja es `Otros`, que solo está en la derecha (`right_only`). Con `how="left"` y `cantones` a la izquierda, el resultado conserva exactamente los 82 cantones y deja fuera `Otros`, que no es un cantón. Con la población en la misma tabla, la tasa es una operación vectorizada:

In [29]:
# Unión definitiva: a cada cantón se le agregan sus casos positivos
cantones_covid = cantones[["provincia", "canton", "codigo_canton", "poblacion"]].merge(
    covid[["codigo_canton", "positivos"]],
    on="codigo_canton",
    how="left"
)

# Casos positivos por cada 100 000 habitantes
cantones_covid["tasa_100k"] = (
    cantones_covid["positivos"] / cantones_covid["poblacion"] * 100000
).round(0)

# Verificación: 82 filas y ningún cantón sin casos
print(len(cantones_covid), cantones_covid["positivos"].isna().sum())

# Los cinco cantones con más casos...
cantones_covid.sort_values("positivos", ascending=False).head()

82 0


,provincia,canton,codigo_canton,poblacion,positivos,tasa_100k
0,San José,San José,101,352381,79939,22685.0
20,Alajuela,Alajuela,201,322143,64702,20085.0
2,San José,Desamparados,103,223226,43283,19390.0
36,Cartago,Cartago,301,165417,33168,20051.0
29,Alajuela,San Carlos,210,198742,32552,16379.0


In [30]:
# ...y los cinco con la tasa más alta: no son los mismos
cantones_covid.sort_values("tasa_100k", ascending=False).head()

,provincia,canton,codigo_canton,poblacion,positivos,tasa_100k
8,San José,Santa Ana,109,58020,15398,26539.0
51,Heredia,Flores,408,22026,5841,26519.0
50,Heredia,Belén,407,23759,5976,25153.0
75,Puntarenas,Garabito,611,26672,6487,24321.0
44,Heredia,Heredia,401,131901,31775,24090.0


San José, Alajuela y Desamparados encabezan el conteo de casos porque son los cantones más poblados; en tasa, los primeros lugares son cantones medianos como Santa Ana, Flores y Belén. Es la misma lección de las densidades: los **conteos absolutos** reflejan sobre todo el tamaño de la población, y para comparar entidades de tamaños distintos se **normalizan** (por población, por área) antes de compararlas o de mapearlas.

Un último detalle: la unión se hizo por código, aunque ambas tablas tienen el nombre del cantón. El ejercicio 7 muestra qué habría pasado al unir por nombre.

Después de una unión conviene verificar siempre dos cosas: la **cantidad de filas** del resultado (¿se perdieron o se duplicaron filas?) y la **cantidad de valores faltantes** en las columnas agregadas, con las herramientas de la sección siguiente.

*Ejercicios de esta sección: ejercicios sobre uniones, en la sección de ejercicios al final del cuaderno.*

## Datos faltantes

Los valores faltantes son parte normal de los datos reales: una estación que no midió, una pregunta sin responder, un registro de GBIF sin año (como en la [parte III](https://gf0657-programacionsig.github.io/2026-ii/estructuras-datos-apis/) de la serie anterior) o, como se acaba de ver, una unión incompleta. pandas los representa con `NaN`.

### Detección

El cuadro 1 del archivo del INEC tiene la población de Costa Rica en todos los censos, desde 1864, con la tasa de crecimiento respecto al censo anterior. El primer censo no tiene censo anterior, y el INEC lo indica con un guion (`-`) en la celda. El argumento `na_values` de `read_excel()` (y de `read_csv()`) indica cuáles textos deben interpretarse como valores faltantes:

In [31]:
# Lectura del cuadro 1: población de Costa Rica en los censos, 1864-2022
censos = pd.read_excel(
    DATOS + "/reResultadosEstimacionPoblacionVivienda2022_3.xlsx",
    sheet_name="1",
    skiprows=6,
    nrows=11,
    header=None,
    names=["anio", "poblacion", "hombres", "mujeres", "tasa_crecimiento"],
    na_values="-"
)

censos

,anio,poblacion,hombres,mujeres,tasa_crecimiento
0,1864,120499,58091,62408,NaN
1,1883,182073,89789,92284,2.17
2,1892,243205,122480,120725,3.22
3,1927,471524,238028,233496,1.89
4,1950,800875,399859,401016,2.30
5,1963,1336274,668957,667317,3.94
6,1973,1871780,938535,933245,3.37
7,1984,2416809,1208216,1208593,2.32
8,2000,3810179,1902614,1907565,2.85
9,2011,4301712,2106063,2195649,1.10


Sin `na_values`, el guion habría convertido toda la columna `tasa_crecimiento` en texto, y ninguna operación numérica habría funcionado con ella: es el problema de las "columnas numéricas leídas como texto" que `info()` ayuda a detectar. Instituciones y programas usan marcadores muy diversos para los valores faltantes (`-`, `ND`, `N/A`, `-9999`, celdas vacías), por lo que hay que revisar la documentación de cada fuente.

El método `isna()` retorna una máscara booleana con `True` donde falta un valor (y `notna()`, lo contrario). Como `True` cuenta como 1, su suma es el conteo de valores faltantes de cada columna:

In [32]:
# Cantidad de valores faltantes por columna
censos.isna().sum()

anio                0
poblacion           0
hombres             0
mujeres             0
tasa_crecimiento    1
dtype: int64

In [33]:
# Filas con valor faltante en la tasa de crecimiento
censos[censos["tasa_crecimiento"].isna()]

,anio,poblacion,hombres,mujeres,tasa_crecimiento
0,1864,120499,58091,62408,NaN


### Los valores faltantes en los cálculos

Las funciones de agregación de pandas **omiten** los valores faltantes: el promedio de la tasa de crecimiento se calcula con los diez valores existentes, y `count()` (a diferencia de `size()` y de `len()`) cuenta solo los valores no nulos:

In [34]:
# El promedio omite el valor faltante: usa 10 valores, no 11
print(censos["tasa_crecimiento"].mean().round(2))
print(censos["tasa_crecimiento"].count())
print(len(censos))

2.46
10
11


En cambio, las operaciones entre columnas **propagan** el valor faltante: cualquier cálculo con `NaN` da `NaN`.

### Descartar o reemplazar

Hay dos maneras básicas de tratar los valores faltantes. `dropna()` descarta las filas que los tienen (en cualquier columna o, con `subset`, en las indicadas):

In [35]:
# Censos con tasa de crecimiento conocida
censos.dropna(subset=["tasa_crecimiento"]).head(3)

,anio,poblacion,hombres,mujeres,tasa_crecimiento
1,1883,182073,89789,92284,2.17
2,1892,243205,122480,120725,3.22
3,1927,471524,238028,233496,1.89


`fillna()` los reemplaza por un valor. Como `sort_values()` o `drop()`, ambos métodos retornan un DataFrame nuevo y dejan el original intacto:

In [36]:
# Reemplazo del valor faltante por cero... ¿es correcto?
censos.fillna(0).head(3)

,anio,poblacion,hombres,mujeres,tasa_crecimiento
0,1864,120499,58091,62408,0.00
1,1883,182073,89789,92284,2.17
2,1892,243205,122480,120725,3.22


El código funciona, pero el resultado afirma algo falso: que la población no creció antes de 1864. El valor no es cero, es **desconocido** (en rigor, no aplicable). Reemplazar valores faltantes es una decisión sobre los datos, no un trámite: es válido cuando se sabe qué significa la ausencia (ej. una categoría sin registros cuyo conteo es, en efecto, cero) y debe documentarse. Ante la duda, es preferible conservar el `NaN`, que pandas sabe omitir, o descartar las filas de forma explícita.

*Ejercicios de esta sección: ejercicios sobre datos faltantes, en la sección de ejercicios al final del cuaderno.*

## Clínica de errores

Los errores típicos de las uniones y de la lectura de archivos, provocados a propósito para aprender a reconocerlos:

In [37]:
# KeyError: la llave de la unión no existe en alguno de los DataFrames
# (provincias no tiene la columna codigo_canton)
cantones.merge(provincias, on="codigo_canton")

KeyError: 'codigo_canton'

In [38]:
# ValueError: "You are trying to merge on int64 and str columns"
# (la llave es un número en un DataFrame y un texto en el otro;
# ocurre cuando un código se lee como texto en uno de los archivos)
codigos_texto = provincias[["codigo_provincia", "poblacion"]].copy()
codigos_texto["codigo_provincia"] = codigos_texto["codigo_provincia"].astype(str)

cantones.merge(codigos_texto, on="codigo_provincia")

ValueError: You are trying to merge on int64 and str columns for key 'codigo_provincia'. If you wish to proceed you should use pd.concat

In [39]:
# ValueError: la hoja no existe en el archivo de Excel
# (el archivo del INEC tiene las hojas "1" a "12")
pd.read_excel(DATOS + "/reResultadosEstimacionPoblacionVivienda2022_3.xlsx", sheet_name="13")

ValueError: Worksheet named '13' not found

El segundo error es frecuente con datos geográficos: los códigos de provincias, cantones o países son números para algunas fuentes y hileras para otras (a veces con ceros a la izquierda, como `"01"`). La solución es convertir una de las dos llaves con `astype()` antes de unir.

Los tres errores anteriores al menos avisan. Los problemas más delicados de este cuaderno son los **silenciosos**, que no producen ningún mensaje: la unión que pierde filas porque las llaves no coinciden, el promedio de densidades que no es una densidad y el `fillna(0)` que inventa un dato. Contra esos solo funciona la verificación: contar filas, contar valores faltantes y contrastar los resultados con una fuente independiente.

## Asistentes de IA para generar y verificar código

Hasta ahora, los asistentes de IA se han usado en el curso para **explicar y depurar**. Según el calendario de la lección de [asistentes de IA](https://gf0657-programacionsig.github.io/2026-ii/asistentes-ia/), a partir de esta semana se usan también para **generar** código de análisis de datos. Tiene sentido hacerlo ahora y no antes: con los fundamentos de Python y de pandas ya es posible leer el código generado, que es la condición para usarlo con responsabilidad. Puede usarse cualquier asistente conversacional, según las guías de [GitHub Copilot](https://gf0657-programacionsig.github.io/2026-ii/copilot/) y [Colab](https://gf0657-programacionsig.github.io/2026-ii/colab/).

### Generar

El asistente no ve los datos: solo conoce lo que el prompt le diga. La estructura de rol, contexto, tarea y formato sigue siendo válida, y el **contexto** se vuelve decisivo: debe describir el DataFrame (nombre, qué representa cada fila y las columnas relevantes con su tipo). Por ejemplo:

> Actúe como asistente de análisis de datos con pandas **(rol)**. Tengo un DataFrame llamado `cantones` con una fila por cada cantón de Costa Rica y las columnas `provincia` (texto), `canton` (texto) y `poblacion` (entero) **(contexto)**. Escriba el código para obtener, de cada provincia, el cantón más poblado y su población **(tarea)**. Use solo pandas, comente cada paso y no invente columnas que no mencioné **(formato)**.

Una forma práctica de dar el contexto es pegar en el prompt la salida de `cantones.dtypes` o de `cantones.head(3)`, siempre que los datos no sean sensibles (los [lineamientos del curso](https://gf0657-programacionsig.github.io/2026-ii/asistentes-ia/) prohíben ingresar datos personales).

### Verificar

El código generado puede ejecutarse sin errores y aun así responder mal la pregunta: es el mismo problema de los errores silenciosos de la clínica, ahora con un autor que se expresa con total seguridad. Antes de incorporar código generado a un trabajo, se verifica con estas cuatro comprobaciones:

1. **Leerlo**: ¿se entiende cada línea? Si usa un método desconocido, se consulta la [documentación de pandas](https://pandas.pydata.org/docs/) o se pide al asistente que lo explique. Código que no se entiende no se entrega.
2. **Ejecutarlo y revisar la forma del resultado**: ¿tiene la cantidad de filas esperada (siete provincias, 82 cantones)? ¿Hay valores faltantes inesperados?
3. **Comprobar un caso a mano**: calcular el resultado de un grupo por otro camino —un filtro y un ordenamiento de la parte I, por ejemplo— y compararlo.
4. **Contrastar con una fuente independiente**: los totales deben coincidir con los publicados (la población del país es 5 044 197; la densidad de las provincias está en `provincias`).

La siguiente celda aplica la comprobación 3 a la tarea del prompt anterior: obtiene el cantón más poblado de una sola provincia con herramientas ya conocidas, para compararlo con lo que produzca el código generado.

In [40]:
# Comprobación a mano para una provincia: el cantón más poblado de Limón
cantones[cantones["provincia"] == "Limón"].sort_values("poblacion", ascending=False)[
    ["provincia", "canton", "poblacion"]
].head(1)

,provincia,canton,poblacion
77,Limón,Pococí,146320


El uso de un asistente para generar código se declara en los trabajos, como pide la lección de asistentes de IA: cuál herramienta, con qué propósito y en cuáles partes.

*Ejercicios de esta sección: ejercicios sobre asistentes de IA, en la sección de ejercicios al final del cuaderno.*

## Resumen

- La **agrupación** sigue la estrategia **dividir, aplicar y combinar**: `df.groupby("columna")["otra"].sum()` resume una columna por cada grupo. `agg()` aplica varias funciones a la vez y `reset_index()` convierte los grupos en una columna.
- Las **razones** (densidades, porcentajes, tasas) no se promedian: se recalculan a partir de los totales de cada grupo.
- `pd.read_excel()` lee hojas de Excel; `sheet_name`, `skiprows`, `nrows`, `header` y `names` delimitan el rectángulo de datos, y `na_values` declara los marcadores de valores faltantes.
- Una **unión** combina dos DataFrames por una **llave** común: `izquierda.merge(derecha, on="llave", how="left")`. Es preferible unir por **código** que por nombre; las llaves de texto se limpian antes con `.str.strip()`.
- El argumento `how` decide qué pasa con las llaves sin pareja (figura 1): `"inner"` descarta esas filas sin aviso, `"left"` conserva todas las de la izquierda con valores faltantes, `"outer"` conserva todas y, con `indicator=True`, indica de dónde viene cada fila. Después de unir se cuentan las filas y los valores faltantes.
- Los **conteos absolutos** reflejan el tamaño de la población: para comparar entidades distintas se calculan **tasas** (ej. casos por 100 000 habitantes).
- Los **valores faltantes** se representan con `NaN`: se detectan con `isna().sum()`, las agregaciones los omiten, `dropna()` los descarta y `fillna()` los reemplaza, lo que es una decisión sobre los datos que debe justificarse.
- Los asistentes de IA se usan desde esta semana para **generar** código: el prompt describe el DataFrame, y el resultado se **verifica** leyéndolo, revisando su forma, comprobando un caso a mano y contrastando con una fuente independiente.

## Ejercicios

Los ejercicios se agrupan según la sección del cuaderno a la que corresponden; se recomienda realizarlos al concluir la sección respectiva. Resuélvalos en este mismo cuaderno (en su copia de Colab o local), después de ejecutar las celdas de las secciones anteriores. Varios incluyen una **celda de verificación**: ejecútela tal cual después de resolver el ejercicio (en la versión publicada muestra un recordatorio, porque los ejercicios no están resueltos). Las [soluciones de estos ejercicios](https://gf0657-programacionsig.github.io/2026-ii/soluciones-pandas-agrupacion-uniones) se publican después de la clase correspondiente.

### Agrupación

1. Cree una Series llamada `viviendas_provincia` con el total de viviendas de cada provincia, calculado a partir de `cantones`. ¿Cuál provincia tiene más viviendas? ¿Y cuál tiene más viviendas desocupadas?

In [41]:
# Celda de verificación del ejercicio 1: ejecútela tal cual
try:
    assert len(viviendas_provincia) == 7, "La Series debe tener un valor por provincia"
    assert viviendas_provincia.loc["Guanacaste"] == 163747, "El total de Guanacaste no es el esperado; revise la columna y la función"
    assert viviendas_provincia.sum() == cantones["viviendas"].sum(), "La suma de los grupos debe coincidir con el total del país"
    print("Ejercicio 1: ¡correcto!")
except NameError:
    print("Aún no se ha definido la Series viviendas_provincia (ejercicio 1).")

Aún no se ha definido la Series viviendas_provincia (ejercicio 1).


2. Con `agg()`, cree un DataFrame llamado `resumen_cantones` con una fila por cantón, a partir de `distritos`, con las columnas `codigo_canton`, `canton`, `distritos` (cantidad de distritos) y `area_mayor_distrito` (área del distrito más extenso). Agrupe por código y nombre, y use `reset_index()`. ¿Cuál es el cantón con más distritos?

In [42]:
# Celda de verificación del ejercicio 2: ejecútela tal cual
try:
    assert resumen_cantones.shape == (82, 4), "El DataFrame debe tener 82 filas y 4 columnas (¿faltó reset_index()?)"
    assert list(resumen_cantones.columns) == ["codigo_canton", "canton", "distritos", "area_mayor_distrito"], \
        "Los nombres o el orden de las columnas no son los esperados"
    assert resumen_cantones["distritos"].sum() == 487, "La suma de distritos debe ser 487"
    assert resumen_cantones.loc[resumen_cantones["distritos"].idxmax(), "canton"] == "Puntarenas", \
        "El cantón con más distritos no es el esperado"
    print("Ejercicio 2: ¡correcto!")
except NameError:
    print("Aún no se ha definido el DataFrame resumen_cantones (ejercicio 2).")

Aún no se ha definido el DataFrame resumen_cantones (ejercicio 2).


3. **Prediga antes de ejecutar**: la columna `promedio_ocupantes` de `cantones` es el promedio de ocupantes por vivienda ocupada de cada cantón. ¿El siguiente programa calcula correctamente el promedio de ocupantes por vivienda ocupada de cada provincia? Escriba su predicción y su razonamiento en una celda de texto; después ejecútelo y compare el resultado con la columna `promedio_ocupantes` de `provincias`.

```python
cantones.groupby("provincia")["promedio_ocupantes"].mean().round(2)
```

<details>
<summary>Después de ejecutar, haga clic aquí para ver la explicación</summary>

No: es la misma trampa de la densidad. El promedio de ocupantes es una razón (personas entre viviendas ocupadas), y promediar las razones de los cantones da el mismo peso a un cantón pequeño que a uno grande. Los resultados se parecen a los del INEC —porque el indicador varía poco entre cantones— pero no coinciden. El cálculo correcto recalcula la razón a partir de los totales de cada provincia, aunque con estos datos solo puede aproximarse (<code>poblacion</code> entre <code>viviendas_ocupadas</code>), porque el INEC calcula el indicador con los ocupantes de las viviendas individuales, que no son exactamente la población total. Que un resultado "se parezca" al correcto no lo valida: puede ser una coincidencia de estos datos.

</details>

### Lectura de Excel

4. La hoja `"5"` del archivo de Excel del INEC contiene los diez cantones con mayor tasa de crecimiento de población entre 2011 y 2022. Ábrala en una hoja de cálculo (o examine el archivo en el [directorio de datos](https://github.com/gf0657-programacionsig/2026-ii/tree/main/datos/inec) del repositorio) para determinar en cuál fila empiezan los datos, y léala en un DataFrame llamado `mayor_crecimiento` con las columnas `canton`, `poblacion_2011`, `poblacion_2022` y `tasa_crecimiento`. Pista: la hoja tiene una quinta columna vacía; el argumento `usecols="A:D"` limita la lectura a las columnas A a D.

In [43]:
# Celda de verificación del ejercicio 4: ejecútela tal cual
try:
    assert mayor_crecimiento.shape == (10, 4), "El DataFrame debe tener 10 filas y 4 columnas; revise skiprows, nrows y usecols"
    assert list(mayor_crecimiento.columns) == ["canton", "poblacion_2011", "poblacion_2022", "tasa_crecimiento"], \
        "Los nombres de las columnas no son los esperados"
    assert mayor_crecimiento.iloc[0]["canton"] == "Talamanca", "La primera fila debe ser Talamanca; revise skiprows"
    assert mayor_crecimiento["tasa_crecimiento"].dtype == "float64", "La tasa debe ser numérica; revise que no se lean notas al pie"
    print("Ejercicio 4: ¡correcto!")
except NameError:
    print("Aún no se ha definido el DataFrame mayor_crecimiento (ejercicio 4).")

Aún no se ha definido el DataFrame mayor_crecimiento (ejercicio 4).


### Uniones

5. El cuadro del ejercicio anterior no indica la provincia de cada cantón. Una `mayor_crecimiento` con las columnas `canton` y `provincia` de `cantones` en un DataFrame llamado `crecimiento_provincia` (aquí se puede unir por nombre, porque los nombres de cantón no se repiten). ¿Cuál provincia tiene más cantones entre los diez de mayor crecimiento? ¿Qué podría explicarlo? Verifique que no se perdieron filas ni quedaron valores faltantes.

In [44]:
# Celda de verificación del ejercicio 5: ejecútela tal cual
try:
    assert len(crecimiento_provincia) == 10, "Deben quedar 10 filas; revise las llaves de la unión"
    assert crecimiento_provincia["provincia"].isna().sum() == 0, "Hay cantones sin provincia: alguna llave no coincidió"
    assert crecimiento_provincia["provincia"].value_counts().idxmax() == "Guanacaste", \
        "La provincia con más cantones no es la esperada"
    print("Ejercicio 5: ¡correcto!")
except NameError:
    print("Aún no se ha definido el DataFrame crecimiento_provincia (ejercicio 5).")

Aún no se ha definido el DataFrame crecimiento_provincia (ejercicio 5).


6. Ahora en el otro sentido: una `cantones` (a la izquierda) con las columnas `canton` y `poblacion_2011` de `mayor_crecimiento`, con `how="left"`, en un DataFrame llamado `cantones_2011`. ¿Cuántas filas tiene el resultado? ¿Cuántos valores faltantes hay en `poblacion_2011` y por qué? Repita la unión con `how="inner"` y compare la cantidad de filas.

In [45]:
# Celda de verificación del ejercicio 6: ejecútela tal cual
try:
    assert len(cantones_2011) == 82, "Con how='left' deben conservarse los 82 cantones"
    assert cantones_2011["poblacion_2011"].isna().sum() == 72, "Deben quedar 72 cantones sin población de 2011"
    print("Ejercicio 6: ¡correcto!")
except NameError:
    print("Aún no se ha definido el DataFrame cantones_2011 (ejercicio 6).")

Aún no se ha definido el DataFrame cantones_2011 (ejercicio 6).


7. Repita la unión de `cantones` con `covid`, pero por **nombre** de cantón (`on="canton"`, `how="left"`, solo las columnas `canton` y `positivos` de `covid`), en un DataFrame llamado `covid_nombre`. ¿Cuántos cantones quedan sin casos positivos y cuáles son? Compare sus nombres en las dos fuentes (por ejemplo, con el DataFrame `diagnostico`) y explique en una celda de texto por qué la unión por código no tuvo ese problema. ¿En cuál región del diagrama de Venn cayeron esos cantones?

In [46]:
# Celda de verificación del ejercicio 7: ejecútela tal cual
try:
    assert len(covid_nombre) == 82, "Con how='left' deben conservarse los 82 cantones"
    assert covid_nombre["positivos"].isna().sum() == 3, "Deben quedar exactamente 3 cantones sin casos: revise la llave y el tipo de unión"
    print("Ejercicio 7: ¡correcto!")
except NameError:
    print("Aún no se ha definido el DataFrame covid_nombre (ejercicio 7).")

Aún no se ha definido el DataFrame covid_nombre (ejercicio 7).


8. Calcule qué porcentaje de la población de su cantón tiene cada distrito del cantón que eligió en la [parte I](https://gf0657-programacionsig.github.io/2026-ii/fundamentos-python/) de la serie *Fundamentos de Python*: una `distritos` con la población de `cantones` (por código, con `suffixes`), cree la columna del porcentaje, filtre su cantón y ordene de mayor a menor. Compruebe que los porcentajes de su cantón suman 100.

### Datos faltantes

9. El método `diff()` de una Series calcula la diferencia entre cada valor y el anterior. Agregue a `censos` una columna llamada `aumento` con el aumento de población respecto al censo anterior. ¿Cuántos valores faltantes tiene la columna y por qué? ¿En cuál año censal se registró el mayor aumento? ¿Sería correcto reemplazar ese valor faltante con `fillna(0)`?

In [47]:
# Celda de verificación del ejercicio 9: ejecútela tal cual
if "aumento" not in censos.columns:
    print("Aún no se ha creado la columna aumento (ejercicio 9).")
else:
    assert censos["aumento"].isna().sum() == 1, "Debe haber exactamente un valor faltante (¿usó fillna()?)"
    assert censos.loc[censos["aumento"].idxmax(), "anio"] == 2000, "El año del mayor aumento no es el esperado"
    print("Ejercicio 9: ¡correcto!")

Aún no se ha creado la columna aumento (ejercicio 9).


### Asistentes de IA

10. Pida a un asistente de IA que genere el código para la tarea del prompt de ejemplo (el cantón más poblado de cada provincia y su población), con la estructura de rol, contexto, tarea y formato. Aplique las cuatro comprobaciones de la sección: lea el código y anote los métodos que no conocía, ejecútelo, revise la forma del resultado, compare el caso de Limón con la celda de comprobación y compare otra provincia por su cuenta. Documente en una celda de texto el asistente, el prompt, el código generado y el resultado de cada comprobación: esa documentación es la **declaración de uso de IA** que piden los [lineamientos del curso](https://gf0657-programacionsig.github.io/2026-ii/asistentes-ia/).

11. Pida ahora al asistente, en la misma conversación, que calcule la densidad de población de cada provincia a partir de `cantones`, **sin advertirle** de la trampa de los promedios. ¿Promedió las densidades de los cantones o dividió los totales? Verifique el resultado contra la columna `densidad` de `provincias`. Si se equivocó, indíquele el error y evalúe su respuesta: ¿corrige el código?, ¿explica bien por qué estaba mal? Recuerde la **adulación** estudiada en la lección de asistentes de IA: que el asistente le dé la razón no demuestra que usted la tenga.

## Referencias bibliográficas

Instituto Nacional de Estadística y Censos. (2023). *Resultados Estimación de Población y Vivienda 2022* [Conjunto de datos]. INEC. https://admin.inec.cr/sites/default/files/2023-11/reResultadosEstimacionPoblacionVivienda2022_3.xlsx
\
\
Kaggle. (s. f.). Data types and missing values. En *Pandas*. Kaggle Learn. Recuperado el 21 de setiembre de 2026, de https://www.kaggle.com/code/residentmario/data-types-and-missing-values
\
\
Kaggle. (s. f.). Grouping and sorting. En *Pandas*. Kaggle Learn. Recuperado el 21 de setiembre de 2026, de https://www.kaggle.com/code/residentmario/grouping-and-sorting
\
\
McKinney, W. (2022). Data aggregation and group operations. En *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media. https://wesmckinney.com/book/data-aggregation
\
\
McKinney, W. (2022). Data wrangling: Join, combine, and reshape. En *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media. https://wesmckinney.com/book/data-wrangling
\
\
Ministerio de Salud. (2022). *Situación Nacional COVID-19: casos positivos acumulados por cantón al 30 de mayo de 2022* [Conjunto de datos]. Ministerio de Salud de Costa Rica. https://github.com/gf0657-programacionsig/2026-ii/tree/main/datos/ministerio-salud
\
\
The pandas development team. (s. f.). *pandas documentation*. Recuperado el 21 de setiembre de 2026, de https://pandas.pydata.org/docs/